### Visualizing the image right before prediction

In [ ]:
import random
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from PIL import Image

from utils import get_dataset_roots, collect_images
from mlep_fast_evaluator import get_mlep_transform


def visualize_random_mlep_preprocessed_image(
    load_size=256,
    crop_size=224,
    no_resize=False,
    no_crop=True,
    seed=None,
):
    """
    Picks one random dataset, one random image, applies the exact MLEP preprocessing,
    and shows:
      1. original image
      2. preprocessed image after de-normalizing for visualization

    The actual tensor printed is the real tensor passed to the model.
    """

    if seed is not None:
        random.seed(seed)

    dataset_roots = get_dataset_roots()

    # Keep only datasets that exist and contain images
    valid_datasets = []
    for dataset_name, dataset_root in dataset_roots.items():
        dataset_root = Path(dataset_root)
        if not dataset_root.exists():
            continue

        samples = collect_images(dataset_root)
        if len(samples) > 0:
            valid_datasets.append((dataset_name, dataset_root, samples))

    if not valid_datasets:
        raise RuntimeError("No valid datasets found. Check your Datasets/ paths.")

    dataset_name, dataset_root, samples = random.choice(valid_datasets)
    sample = random.choice(samples)

    img_path = Path(sample["path"])
    original_img = Image.open(img_path).convert("RGB")

    transform = get_mlep_transform(
        load_size=load_size,
        crop_size=crop_size,
        no_resize=no_resize,
        no_crop=no_crop,
    )

    preprocessed_tensor = transform(original_img)          # [3, H, W]
    model_input_tensor = preprocessed_tensor.unsqueeze(0)  # [1, 3, H, W] -> this is passed to model

    # De-normalize only for visualization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    raw_vis_tensor = preprocessed_tensor.cpu().clamp(0, 1)
    raw_vis_img = raw_vis_tensor.permute(1, 2, 0).numpy()

    denorm_tensor = preprocessed_tensor.cpu() * std + mean
    denorm_tensor = denorm_tensor.clamp(0, 1)
    denorm_img = denorm_tensor.permute(1, 2, 0).numpy()

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(original_img)
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(raw_vis_img)
    plt.title("Raw normalized tensor\nclipped for display")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(denorm_img)
    plt.title("After preprocessing\n(de-normalized)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    return {
        "dataset_name": dataset_name,
        "dataset_root": str(dataset_root),
        "image_id": sample["id"],
        "image_path": str(img_path),
        "true_label": sample["true_label"],
        "split": sample["split"],
        "model_input_tensor": model_input_tensor,
    }


sample_info = visualize_random_mlep_preprocessed_image()

### Finetuning and comparing results

In [1]:
from utils import get_dataset_roots

get_dataset_roots()

{'20K_real_and_deepfake_images': WindowsPath('c:/Extra_D/Ali_Master/Term 3/Computer Vision/Project/DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE/Datasets/20K_real_and_deepfake_images'),
 'DeepDetect-2025': WindowsPath('c:/Extra_D/Ali_Master/Term 3/Computer Vision/Project/DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE/Datasets/DeepDetect-2025/ddata'),
 'Deepfake-vs-Real-v2': WindowsPath('c:/Extra_D/Ali_Master/Term 3/Computer Vision/Project/DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE/Datasets/Deepfake-vs-Real-v2'),
 'gravex200k': WindowsPath('c:/Extra_D/Ali_Master/Term 3/Computer Vision/Project/DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE/Datasets/gravex200k/my_real_vs_ai_dataset/my_real_vs_ai_dataset'),
 'nuriachandra_Deepfake-Eval-2024': WindowsPath('c:/Extra_D/Ali_Master/Term 3/Computer Vision/Project/DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE/Datasets/nuriachandra_Deepfake-Eval-2024')}

In [2]:
from finetune_mlep import finetune_mlep

result = finetune_mlep(
    dataset_name="gravex200k",
    weights_path="MLEP/pretrained/model_epoch_best.pth",
    max_per_class=20000,
    epochs=30,
    batch_size=32,
    lr=1e-5,
    num_workers=0,
    amp=False,
    freeze_backbone=False,
)

Starting MLEP fine-tuning...
Device: cuda:0
Dataset: gravex200k
Dataset root: c:\Extra_D\Ali_Master\Term 3\Computer Vision\Project\DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE\Datasets\gravex200k\my_real_vs_ai_dataset\my_real_vs_ai_dataset
Total selected: 40000 {0: 20000, 1: 20000}
Train: 32000 {0: 16000, 1: 16000}
Val: 8000 {0: 4000, 1: 4000}
Loading MLEP model...
Loaded MLEP model weights: MLEP\pretrained\model_epoch_best.pth
Trainable parameters: 1441217

Epoch 1/30


c:\Extra_D\Ali_Master\Term 3\Computer Vision\Project\DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE\finetune_mlep.py:351: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=amp and device.type == "cuda")


train:   0%|          | 0/1000 [00:00<?, ?it/s]

c:\Extra_D\Ali_Master\Term 3\Computer Vision\Project\DeepFakeDetectionAndPrevention_AKA_DEEPFAKEPOLICE\finetune_mlep.py:197: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


val:   0%|          | 0/250 [00:00<?, ?it/s]

...

val:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss: 0.587794 metrics: {'accuracy': 0.68003125, 'precision': 0.6799075635500593, 'recall': 0.680375, 'f1': 0.6801412014619974} counts: {'TP': 10886, 'TN': 10875, 'FP': 5125, 'FN': 5114}
Val   loss: 0.629721 metrics: {'accuracy': 0.630875, 'precision': 0.6586844498332828, 'recall': 0.54325, 'f1': 0.5954240306891355} counts: {'TP': 2173, 'TN': 2874, 'FP': 1126, 'FN': 1827}

Epoch 30/30


train:   0%|          | 0/1000 [00:00<?, ?it/s]

val:   0%|          | 0/250 [00:00<?, ?it/s]

Train loss: 0.585549 metrics: {'accuracy': 0.68178125, 'precision': 0.6818380743982495, 'recall': 0.681625, 'f1': 0.6817315205500859} counts: {'TP': 10906, 'TN': 10911, 'FP': 5089, 'FN': 5094}
Val   loss: 0.631359 metrics: {'accuracy': 0.632125, 'precision': 0.6648985959438377, 'recall': 0.53275, 'f1': 0.5915336571825122} counts: {'TP': 2131, 'TN': 2926, 'FP': 1074, 'FN': 1869}

Done.
Best checkpoint: Finetuned\MLEP\MLEP_finetuned_gravex200k_best.pth
Last checkpoint: Finetuned\MLEP\MLEP_finetuned_gravex200k_last.pth
History: Finetuned\MLEP\MLEP_finetuned_gravex200k_history.json


In [3]:
from mlep_fast_evaluator import load_mlep_model, evaluate_all_datasets_mlep_fast

model, device = load_mlep_model(
    repo_dir="MLEP",
    weights_path=result["best_checkpoint"],
)

results = evaluate_all_datasets_mlep_fast(
    model=model,
    device=device,
    model_name="finetuned_Frozen_MLEP_gravex200k_20k_per_class_full",
    batch_size=64,
    checkpoint_every=50,
    num_workers=0,
    # max_batches=3,   # smoke test first
    threshold=0.5,
)

Loaded MLEP model weights: Finetuned\MLEP\MLEP_finetuned_gravex200k_best.pth

[1/5] Running dataset: 20K_real_and_deepfake_images
[1/5] Running 20K_real_and_deepfake_images: 17354 remaining


[1/5] 20K_real_and_deepfake_images:   0%|          | 0/272 [00:00<?, ?it/s]


[2/5] Running dataset: DeepDetect-2025
[2/5] Running DeepDetect-2025: 112185 remaining


[2/5] DeepDetect-2025:   0%|          | 0/1753 [00:00<?, ?it/s]


[3/5] Running dataset: Deepfake-vs-Real-v2
[3/5] Running Deepfake-vs-Real-v2: 19291 remaining


[3/5] Deepfake-vs-Real-v2:   0%|          | 0/302 [00:00<?, ?it/s]


[4/5] Running dataset: gravex200k
[4/5] Running gravex200k: 200000 remaining


[4/5] gravex200k:   0%|          | 0/3125 [00:00<?, ?it/s]


[5/5] Running dataset: nuriachandra_Deepfake-Eval-2024
[5/5] Running nuriachandra_Deepfake-Eval-2024: 1952 remaining


[5/5] nuriachandra_Deepfake-Eval-2024:   0%|          | 0/31 [00:00<?, ?it/s]

### 